# Browse DestinE Climate DT Data with Polytope

This notebook demonstrates how to browse DestinE Climate Digital Twin data
using Polytope and `earthkit-data`, for the **ICON**, **IFS-FESOM**,
and **IFS-NEMO** models.

> **Server addresses:** ICON & IFS-FESOM use LUMI, IFS-NEMO uses MN5.
> **Activity:** `'baseline'` for hist/cont; `'projections'` for SSP3-7.0.
> **Streams:** Hourly `clte` has instantaneous params (134 SP, 165 10U, 166 10V).
> For precipitation and monthly-mean temperature use `clmn` +
> `PolytopeZarrStore` (see `03_lazy_browse_portfolio.ipynb`).

See `docs/polytope_setup.md` for prerequisites and setup.

## 1. Imports & Configuration

In [ ]:
import logging, warnings, os
import earthkit.data

for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)

LUMI_ADDRESS = "polytope.lumi.apps.dte.destination-earth.eu"
MN5_ADDRESS  = "polytope.mn5.apps.dte.destination-earth.eu"
COLLECTION   = "destination-earth"

def get_address(model: str) -> str:
    return MN5_ADDRESS if model == "IFS-NEMO" else LUMI_ADDRESS

print(f"earthkit-data {earthkit.data.__version__}")

## 2. Quick Connection Test

In [ ]:
print("Testing LUMI (ICON historical)...")
try:
    test = earthkit.data.from_source(
        "polytope", COLLECTION,
        {'activity': 'baseline', 'class': 'd1', 'dataset': 'climate-dt',
         'date': '20000615', 'experiment': 'hist', 'expver': '0001',
         'generation': '2', 'levtype': 'sfc', 'model': 'ICON',
         'param': '134', 'realization': '1', 'resolution': 'standard',
         'stream': 'clte', 'time': '0000', 'type': 'fc'},
        address=LUMI_ADDRESS, stream=False)
    arr = test.to_numpy()
    print(f"  OK LUMI - {arr.size} values, shape={arr.shape}")
except Exception as e:
    print(f"  LUMI failed: {e}")

print("Testing MN5 (IFS-NEMO SSP3-7.0)...")
try:
    test2 = earthkit.data.from_source(
        "polytope", COLLECTION,
        {'activity': 'projections', 'class': 'd1', 'dataset': 'climate-dt',
         'date': '20200102', 'experiment': 'SSP3-7.0', 'expver': '0001',
         'generation': '2', 'levtype': 'sfc', 'model': 'IFS-NEMO',
         'param': '134', 'realization': '1', 'resolution': 'standard',
         'stream': 'clte', 'time': '0100', 'type': 'fc'},
        address=MN5_ADDRESS, stream=False)
    arr = test2.to_numpy()
    print(f"  OK MN5 - {arr.size} values, shape={arr.shape}")
except Exception as e:
    print(f"  MN5 failed: {e}")

## 3. Fetch & Plot Hourly Data (clte stream)

Climate DT data uses the HEALPix grid. Use `data.to_numpy()` +
`healpy.mollview(field, nest=True, flip='geo')` for correct
visualization. Avoid `ax.scatter()` or xarray `.plot()` on HEALPix
data.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

def healpy_plot(ek_data, var_idx=0, title="", cmap="viridis"):
    """Plot HEALPix GRIB data using healpy.mollview."""
    arr = ek_data.to_numpy()
    if arr.ndim == 2:
        if var_idx >= arr.shape[0]:
            print(f"var_idx={var_idx} out of range (max {arr.shape[0]-1})")
            return
        field = arr[var_idx, :]
    else:
        field = arr
    n_valid = np.sum(~np.isnan(field))
    n_nan = np.sum(np.isnan(field))
    print(f"Pixels: {field.size:,} total, {n_valid:,} valid, {n_nan:,} NaN")
    if n_valid == 0:
        print("ALL NaN - no data to plot.")
        return
    hp.mollview(field, title=title, cmap=cmap, nest=True, flip='geo')
    plt.show()

In [ ]:
icon_req = {
    'activity': 'baseline', 'class': 'd1', 'dataset': 'climate-dt',
    'date': '20000615', 'experiment': 'hist', 'expver': '0001',
    'generation': '2', 'levtype': 'sfc', 'model': 'ICON',
    'param': '134/165/166', 'realization': '1', 'resolution': 'standard',
    'stream': 'clte', 'time': '0000/1200', 'type': 'fc'
}
print("Fetching ICON historical SP/10U/10V...")
try:
    icon_data = earthkit.data.from_source(
        "polytope", COLLECTION, icon_req,
        address=get_address('ICON'), stream=False)
    print(f"Got {icon_data.to_numpy().shape[0]} fields")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
if "icon_data" in dir():
    healpy_plot(icon_data, var_idx=0,
                title="ICON hist - Surface Pressure - 2000-06-15 00:00",
                cmap="viridis")

## 4. Request Builder Helper

In [ ]:
def build_request(model="ICON", experiment="hist", date="20000615",
                  time="0000", param="134", levtype="sfc",
                  resolution="standard", stream="clte", **kwargs):
    """Build a Polytope request dict. Auto-sets activity from experiment."""
    activity = "baseline" if experiment in ("hist", "cont") else "projections"
    req = {
        'activity': activity, 'class': 'd1', 'dataset': 'climate-dt',
        'date': date, 'experiment': experiment, 'expver': '0001',
        'generation': '2', 'levtype': levtype, 'model': model,
        'param': param, 'realization': '1', 'resolution': resolution,
        'stream': stream, 'time': time, 'type': 'fc'
    }
    req.update(kwargs)
    return req

ex = build_request(model="IFS-FESOM", experiment="SSP3-7.0",
                   date="20200615", time="0000/0600/1200/1800",
                   param="134/165/166")
for k, v in ex.items():
    print(f"  {k}: {v}")

## 5. Monthly Data (clmn stream)

Precipitation (`tp` / `avg_tprate`) and monthly temperature (`avg_2t`)
are only in the `clmn` stream. Use `PolytopeZarrStore` -
see `03_lazy_browse_portfolio.ipynb`.

Or via CLI:
```bash
python get-data/download_destine.py --model IFS-FESOM --experiment hist \
    --date 20100601 --param tp --stream clmn
```

## 6. Diagnostics
Run this if plots show no data.

In [ ]:
import sys
print("=== Diagnostics ===")
print(f"Python: {sys.version}")
print(f"earthkit-data: {earthkit.data.__version__}")

apirc = os.path.expanduser("~/.polytopeapirc")
if os.path.exists(apirc):
    print(f"~/.polytopeapirc exists ({os.path.getsize(apirc)} bytes)")
else:
    print("~/.polytopeapirc MISSING")

print("Quick fetch test:")
try:
    for ln in ("polytope", "polytope.api", "earthkit.data"):
        logging.getLogger(ln).setLevel(logging.WARNING)
    d = earthkit.data.from_source(
        "polytope", COLLECTION,
        {'activity':'baseline','class':'d1','dataset':'climate-dt',
         'date':'20000615','experiment':'hist','expver':'0001',
         'generation':'2','levtype':'sfc','model':'ICON',
         'param':'134','realization':'1','resolution':'standard',
         'stream':'clte','time':'0000','type':'fc'},
        address=LUMI_ADDRESS, stream=False)
    print(f"OK - {d.to_numpy().size} values")
except Exception as e:
    print(f"Failed: {e}")

## Notes & Best Practices
- **HEALPix**: Use `hp.mollview(field, nest=True, flip='geo')`
- **Servers**: ICON/IFS-FESOM -> LUMI, IFS-NEMO -> MN5
- **Streams**: `clte` = hourly instantaneous; `clmn` = monthly means
- **Diagnostics**: Run `store.verify()` in `03_lazy_browse_portfolio.ipynb`
- **Docs**: `docs/polytope_setup.md`, `docs/polytope_usage.md`